In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from secml.utils import fm
from secml import settings

import numpy as np

from robustbench.utils import load_model

import matplotlib.pyplot as plt

import torchattacks

from tqdm import tqdm

from nonconformist.cp import IcpRegressor
from nonconformist.nc import RegressorNc, QuantileRegErrFunc

2026-07-17 15:56:33.868544: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-17 15:56:33.951833: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-17 15:56:35.925568: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# DATASET

In [2]:
transform=T.ToTensor()

dataset=torchvision.datasets.CIFAR10(
    root="/home/acarlevaro/Sources/albi/data",
    train=True,
    download=True,
    transform=transform
)

In [3]:
idx=torch.randperm(len(dataset))

idx_cal=idx[:1000]
idx_test=idx[1000:2000]

# LOAD RB MODEL

In [4]:
output_dir = fm.join(settings.SECML_MODELS_DIR, 'robustbench')

model_rb = load_model(
    model_name="Wang2023Better_WRN-70-16", #Wang2023Better_WRN-70-16, Ding2020MMA
    norm="L2",
    model_dir=output_dir
)

model_rb.eval()

model_rb.cuda()

DMWideResNet(
  (init_conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer): Sequential(
    (0): _BlockGroup(
      (block): Sequential(
        (0): _Block(
          (batchnorm_0): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu_0): SiLU()
          (conv_0): Conv2d(16, 256, kernel_size=(3, 3), stride=(1, 1), bias=False)
          (batchnorm_1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu_1): SiLU()
          (conv_1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (shortcut): Conv2d(16, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (1): _Block(
          (batchnorm_0): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu_0): SiLU()
          (conv_0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), bias=False)
          (batchnorm

### Rregresion wrapper

In [9]:
class RobustBenchProbabilityRegressor(nn.Module):

    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier


    def forward(self, x, y):

        logits = self.classifier(x)

        probs = F.softmax(logits, dim=1)

        idx = torch.arange(
            len(y),
            device=x.device
        )

        p_true = probs[idx,y]

        return p_true

In [10]:
regressor = RobustBenchProbabilityRegressor(
    model_rb
).cuda()

### Get probability targets

In [13]:
def get_probability_targets(model,dataset,indices):

    X=[]
    y=[]
    p=[]


    with torch.no_grad():

        for i in indices:

            img,label=dataset[i]

            img=img.unsqueeze(0).cuda()
            label=torch.tensor(
                [label],
                device="cuda"
            )


            prob=model(
                img,
                label
            )


            X.append(img.cpu())
            y.append(label.cpu())
            p.append(prob.cpu())


    return (
        torch.cat(X),
        torch.cat(y),
        torch.cat(p)
    )

In [12]:
X_cal, y_cal, p_cal = get_probability_targets(regressor,dataset,idx_cal)

X_test, y_test, p_test = get_probability_targets(regressor,dataset,idx_test)

# ATTACKS

In [14]:
def pgd_attack(
    model,
    X,
    y,
    eps=0.5,
    alpha=0.005,
    steps=100,
    device="cuda",
    batch_size = 32
):

    model.eval()

    atk = torchattacks.PGD(
        model,
        eps=eps,
        alpha=alpha,
        steps=steps,
        random_start=True
    )

    X_adv = []

    for i in tqdm(range(0, len(X), batch_size), desc="Running PGD", leave=True):

        x_batch = X[i:i+batch_size].to(device)
        y_batch = y[i:i+batch_size].to(device)

        adv = atk(
            x_batch,
            y_batch
        )

        X_adv.append(
            adv.detach().cpu()
        )

    return torch.cat(X_adv)

### PGD example

In [15]:
X_cal_adv = pgd_attack(
    model_rb,
    X_cal,
    y_cal,
    eps=0.5,
    alpha=0.001,
    steps=1
)


X_test_adv = pgd_attack(
    model_rb,
    X_test,
    y_test,
    eps=0.5,
    alpha=0.001,
    steps=1
)

Running PGD: 100%|██████████| 32/32 [00:02<00:00, 11.71it/s]


#### Get true probabilites

In [16]:
def get_true_probabilities(
    classifier,
    X,
    y,
    device="cuda"
):

    classifier.eval()

    probs_all=[]

    batch_size=64

    with torch.no_grad():

        for i in range(0,len(X),batch_size):

            x=X[i:i+batch_size].to(device)
            labels=y[i:i+batch_size].to(device)

            logits=classifier(x)

            probs=torch.softmax(
                logits,
                dim=1
            )

            p_true=probs[
                torch.arange(len(labels),device=device),
                labels
            ]

            probs_all.append(
                p_true.cpu()
            )

    return torch.cat(probs_all)

In [17]:
y_cal_reg_adv = get_true_probabilities(
    model_rb,
    X_cal_adv,
    y_cal
)


y_test_reg_adv = get_true_probabilities(
    model_rb,
    X_test_adv,
    y_test
)

# CQR

### Construct the dataset

In [18]:
from torch.utils.data import Dataset, DataLoader


class ProbabilityRegressionDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y.float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y[idx]
        )

In [19]:
train_dataset = ProbabilityRegressionDataset(
    X_cal,
    p_cal
)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

### Quantile regressor (RESNET-18)

In [20]:
import torchvision.models as models


class QuantileRegressor(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = models.resnet18(
            weights=None
        )

        self.model.fc = nn.Linear(
            self.model.fc.in_features,
            2
        )


    def forward(self,x):

        out = self.model(x)

        return out

In [21]:
def pinball_loss(pred, target, quantile):

    error = target - pred

    return torch.mean(
        torch.maximum(
            quantile*error,
            (quantile-1)*error
        )
    )


def cqr_loss(pred, target):

    loss_low = pinball_loss(
        pred[:,0],
        target,
        q_low
    )

    loss_high = pinball_loss(
        pred[:,1],
        target,
        q_high
    )

    return loss_low + loss_high

#### Train the quantile regressor

In [22]:
alpha=0.1

q_low=alpha/2
q_high=1-alpha/2

In [23]:
device="cuda"


qmodel = QuantileRegressor().to(device)


optimizer = torch.optim.Adam(
    qmodel.parameters(),
    lr=1e-4
)


epochs=50


for epoch in range(epochs):

    qmodel.train()

    total_loss=0


    for X_batch,y_batch in train_loader:

        X_batch=X_batch.to(device)
        y_batch=y_batch.to(device)


        pred=qmodel(X_batch)


        loss=cqr_loss(
            pred,
            y_batch
        )


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        total_loss += loss.item()


    print(
        epoch,
        total_loss/len(train_loader)
    )

0 0.27036473504267633
1 0.09659386589191854
2 0.0908224512822926
3 0.07872963417321444
4 0.07715785678010434
5 0.07271759666036814
6 0.06712549296207726
7 0.06847019621636719
8 0.06272463139612228
9 0.06106825824826956
10 0.06134428211953491
11 0.053200572496280074
12 0.053767086821608245
13 0.04853144905064255
14 0.050265716505236924
15 0.04833685723133385
16 0.04368472029455006
17 0.04192385612986982
18 0.04104781860951334
19 0.03621464065508917
20 0.035295999608933926
21 0.034685309103224427
22 0.03218504134565592
23 0.03227147692814469
24 0.02845761040225625
25 0.026111154176760465
26 0.025350226496811956
27 0.025062690081540495
28 0.02220751903951168
29 0.022208311944268644
30 0.02161744306795299
31 0.021353881515096873
32 0.02135005869786255
33 0.0237188394239638
34 0.020642277435399592
35 0.0191226641763933
36 0.018945176911074668
37 0.02039758733008057
38 0.020821143116336316
39 0.01839430778636597
40 0.01625759925809689
41 0.01711875619366765
42 0.015263078646967188
43 0.01604

##### Control the output

In [23]:
qmodel.eval()

with torch.no_grad():

    pred=qmodel(
        X_test[:10].to(device)
    )


print(pred)

tensor([[0.8597, 1.2008],
        [0.8604, 1.1159],
        [0.9451, 1.1772],
        [0.8061, 1.1104],
        [0.9092, 1.0205],
        [0.7965, 1.0601],
        [0.9036, 1.0053],
        [0.9174, 1.5465],
        [0.8226, 1.0824],
        [0.9122, 0.9950]], device='cuda:0')


#### CQR wrapper

In [24]:
class CQRWrapper:

    def __init__(self, model, device):
        self.model=model
        self.device=device


    def predict(self,X):

        if isinstance(X,np.ndarray):
            X=torch.tensor(X).float()

        X=X.to(self.device)


        with torch.no_grad():

            pred=self.model(X)


        return pred.cpu().numpy()

#### Romano CQR

In [27]:
cp_model=CQRWrapper(
    qmodel,
    device
)

In [28]:
nc = RegressorNc(
    cp_model,
    QuantileRegErrFunc()
)

icp = IcpRegressor(nc)

In [28]:
p_cal_adv = get_true_probabilities(
    model_rb,
    X_cal_adv,
    y_cal
)

p_test_adv = get_true_probabilities(
    model_rb,
    X_test_adv,
    y_test
)

## PERCP

In [25]:
qmodel.eval()

with torch.no_grad():

    pred_cal_adv = qmodel(
        X_cal_adv.to(device)
    )
    
with torch.no_grad():

    pred_test_adv = qmodel(
        X_test_adv.to(device)
    )

In [26]:
lower_cal = pred_cal_adv[:,0]
upper_cal = pred_cal_adv[:,1]

lower_test = pred_test_adv[:,0]
upper_test = pred_test_adv[:,1]

In [29]:
scores_adv = torch.maximum(
    lower_cal - p_cal_adv.to(device),
    p_cal_adv.to(device) - upper_cal
)

In [30]:

qhat_percp = torch.quantile(
    scores_adv,
    1-alpha
)

In [31]:
lo_percp = lower_test - qhat_percp
hi_percp = upper_test + qhat_percp

In [42]:
print(scores_clean)

tensor([-3.0333e-02, -9.2002e-02,  6.4250e-02, -1.5851e-02, -3.0021e-02,
        -4.5836e-02, -2.3368e-02, -1.6106e-01, -8.3261e-03, -7.8677e-02,
         2.8230e-03, -5.8878e-02, -1.0524e-01, -1.4058e-01, -5.8969e-02,
         1.3413e-02,  1.5570e-02,  3.2045e-02, -6.0985e-03, -4.0941e-02,
        -6.2471e-02, -4.0967e-02, -8.2333e-02, -2.6214e-02, -4.2069e-02,
         3.3926e-02, -1.2929e-02,  4.3287e-03, -9.0977e-02, -1.0833e-02,
        -1.5750e-02, -1.5092e-02, -9.3545e-02, -3.4940e-02, -2.7076e-02,
        -1.5951e-02, -2.5192e-02, -3.5032e-02, -6.2905e-02, -5.0051e-02,
        -3.9012e-02,  1.0929e-02, -8.4334e-02,  3.2685e-02, -2.0498e-02,
        -3.8569e-02, -2.0626e-02, -3.0957e-02, -4.5576e-02, -8.0291e-03,
        -7.4041e-02, -9.4119e-02, -3.2110e-02, -5.1955e-02, -9.2958e-02,
        -2.9376e-02, -3.6161e-02, -8.1408e-02,  1.0042e-02, -3.1997e-02,
        -1.3262e-02, -3.0463e-02, -1.3980e-01, -1.7453e-02, -3.8459e-02,
         8.8421e-03, -1.1019e-02, -7.4840e-02, -4.1

# Vanilla

In [32]:
qmodel.eval()

with torch.no_grad():

    pred_cal = qmodel(
        X_cal.to(device)
    )

lower_cal = pred_cal[:,0]
upper_cal = pred_cal[:,1]

In [33]:
p_cal = p_cal.to(device)

scores_clean = torch.maximum(
    lower_cal - p_cal,
    p_cal - upper_cal
)

qhat_vanilla = torch.quantile(
    scores_clean,
    1-alpha
)

# Test

In [34]:
with torch.no_grad():

    pred_test_adv = qmodel(
        X_test_adv.to(device)
    )


lower_test_adv = pred_test_adv[:,0]
upper_test_adv = pred_test_adv[:,1]

In [35]:
p_test_adv = p_test_adv.to(device)

In [36]:
lo_vanilla = lower_test - qhat_vanilla
hi_vanilla = upper_test + qhat_vanilla

In [37]:
coverage_vanilla = (
    (p_test_adv >= lo_vanilla) &
    (p_test_adv <= hi_vanilla)
).float().mean()

size_vanilla = (
    hi_vanilla - lo_vanilla
).mean()

In [38]:
lo_percp = lower_test - qhat_percp
hi_percp = upper_test + qhat_percp

In [39]:
coverage_percp = (
    (p_test_adv >= lo_percp) &
    (p_test_adv <= hi_percp)
).float().mean()

size_percp = (
    hi_percp - lo_percp
).mean()

In [40]:
print(
    f"Vanilla CQR | "
    f"Coverage: {coverage_vanilla.item()*100:.2f}% | "
    f"Avg size: {size_vanilla.item():.4f}"
)


print(
    f"PERCP       | "
    f"Coverage: {coverage_percp.item()*100:.2f}% | "
    f"Avg size: {size_percp.item():.4f}"
)

Vanilla CQR | Coverage: 2.50% | Avg size: 0.3991
PERCP       | Coverage: 90.80% | Avg size: 2.0213
